<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Developing Applications with Apache Spark</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">4. Working with Complex Data</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</p>

# 4.3 Lab - Working with Complex Data Types in E-Commerce Data
### Duration - 25 Minutes

In this lab, you will analyze the `raw_franchise_sales` table created during the required setup - Bakehouse franchise data where the location details, payment methods, and recent sales arrive as JSON strings. You will convert the JSON to structured types and analyze the nested sales data.

## Learning Objectives

**By the end of this lab, you will be able to:**

- Inspect raw data containing JSON strings and infer schemas from sample values
- Convert JSON strings to structured types with `from_json()`
- Unnest arrays with `explode()` and analyze the results
- Aggregate values into collections with `collect_set()`

## REQUIRED - SELECT CLASSIC COMPUTE

<div style="font-size: 1em; border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<div>
<strong style="color: #c62828">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333">Before starting this notebook, select your <strong>Classic Compute</strong> cluster from the compute drop-down at the top-right of the notebook. Be aware that <strong>Serverless</strong> is selected by default.</p>
<ul style="margin: 12px 0 0 16px; color: #333">
<li>If your cluster is not shown in the drop-down, select <strong>More</strong>, then choose your cluster in the <strong>Attach to an existing compute resource</strong> dialog.</li>
<li>If your cluster has terminated, open <strong>Compute</strong> in the left navigation, start your cluster, then return here and attach it.</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333"><strong>NOTE:</strong> This course was developed and tested on <strong>Databricks Runtime 17.3 LTS (Apache Spark 4.0)</strong>. Several demonstrations reference the Spark UI and query execution plans, which are not exposed on Serverless compute.</p>
</div>
</div>
</div>

## A. Classroom Setup

Run the following cell to configure your working environment for this course. It validates that the **01_Demo_Required_Setup** notebook has been run (including the raw complex data tables), sets your default catalog, and sets the default schema to `spark_dev`.

<div style="font-size: 1em; border-left: 4px solid #ffc107; background: #fffde7; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<span style="font-size: 24px">&#127919;</span>
<div>
<strong style="color: #ff8f00; font-size: 1.1em">How This Lab Works</strong>
<p style="margin: 8px 0 0 0; color: #333">Each exercise has a code cell containing <code>FILL_IN</code> placeholders for you to complete and run. If you get stuck, expand the <strong>Expand for Solution</strong> bar beneath the exercise to compare your approach.</p>
</div>
</div>
</div>

In [0]:
%run ../Includes/Classroom-Setup-4

## B. Load and Inspect the Raw Data

First, let's see what we are working with.

### Exercise B1: Read and Inspect

Read the `raw_franchise_sales` table into a DataFrame named `raw_sales_df`, print its schema, and display the first 5 rows. Note which columns hold JSON strings.

In [0]:
# B1. Read the Raw Franchise Sales Table (Your Turn)
# Read the raw franchise sales table
raw_sales_df = FILL_IN

# Examine the schema
FILL_IN

# Display the first 5 rows
FILL_IN

<details>
<summary style="font-size: 1em; cursor: pointer; list-style: none; user-select: none">
<div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: center; gap: 12px">
<span>&#x25B6;</span>
<strong style="font-size: 1.1em; color: #1B5162">Expand for Solution</strong>
</div></div></summary>
<div style="border-left: 4px solid #1B5162; background: transparent; padding: 0 20px 16px 20px; border-radius: 0 0 4px 4px; margin: -16px 0 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<span style="visibility: hidden">&#x25B6;</span>
<div>
<pre style="background:#2d2d2d;border:1px solid #444;color:#cccccc;border-radius:8px;padding:16px;margin:16px 0;overflow-x:auto;font-family:Consolas,Monaco,monospace;font-size:14px;line-height:1.5"><code><span style="color: #999"># Read the raw franchise sales table</span>
<span style="color: #CCC">raw_sales_df</span> <span style="color: #67CDCC">=</span> <span style="color: #CCC">spark</span><span style="color: #67CDCC">.</span><span style="color: #CCC">read</span><span style="color: #67CDCC">.</span><span style="color: #CCC">table(</span><span style="color: #7EC699">"raw_franchise_sales"</span><span style="color: #CCC">)</span>
<span style="color: #999"># Examine the schema</span>
<span style="color: #CCC">raw_sales_df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">printSchema()</span>
<span style="color: #999"># Display the first 5 rows</span>
<span style="color: #CCC">display(raw_sales_df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">limit(</span><span style="color: #F08D49">5</span><span style="color: #CCC">))</span></code></pre>
</div></div></div>
</details>

## C. Convert JSON Strings to Structured Types

The `location`, `payment_methods`, and `recent_sales` columns are JSON strings. Convert them to structured types so we can analyze them.

### Exercise C1: Sample the JSON Columns

Collect one sample value from each of the three JSON columns into the variables `location_sample`, `payment_methods_sample`, and `recent_sales_sample`, and print them.

In [0]:
# C1. Sample the JSON Columns (Your Turn)
location_sample = raw_sales_df.select("location").limit(1).collect()[0][0]
payment_methods_sample = FILL_IN
recent_sales_sample = FILL_IN

print(f"location:        {location_sample}")
print(f"payment_methods: {payment_methods_sample}")
print(f"recent_sales:    {recent_sales_sample[:120]}...")

<details>
<summary style="font-size: 1em; cursor: pointer; list-style: none; user-select: none">
<div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: center; gap: 12px">
<span>&#x25B6;</span>
<strong style="font-size: 1.1em; color: #1B5162">Expand for Solution</strong>
</div></div></summary>
<div style="border-left: 4px solid #1B5162; background: transparent; padding: 0 20px 16px 20px; border-radius: 0 0 4px 4px; margin: -16px 0 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<span style="visibility: hidden">&#x25B6;</span>
<div>
<pre style="background:#2d2d2d;border:1px solid #444;color:#cccccc;border-radius:8px;padding:16px;margin:16px 0;overflow-x:auto;font-family:Consolas,Monaco,monospace;font-size:14px;line-height:1.5"><code><span style="color: #CCC">location_sample</span> <span style="color: #67CDCC">=</span> <span style="color: #CCC">raw_sales_df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">select(</span><span style="color: #7EC699">"location"</span><span style="color: #CCC">)</span><span style="color: #67CDCC">.</span><span style="color: #CCC">limit(</span><span style="color: #F08D49">1</span><span style="color: #CCC">)</span><span style="color: #67CDCC">.</span><span style="color: #CCC">collect()[</span><span style="color: #F08D49">0</span><span style="color: #CCC">][</span><span style="color: #F08D49">0</span><span style="color: #CCC">]</span>
<span style="color: #CCC">payment_methods_sample</span> <span style="color: #67CDCC">=</span> <span style="color: #CCC">raw_sales_df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">select(</span><span style="color: #7EC699">"payment_methods"</span><span style="color: #CCC">)</span><span style="color: #67CDCC">.</span><span style="color: #CCC">limit(</span><span style="color: #F08D49">1</span><span style="color: #CCC">)</span><span style="color: #67CDCC">.</span><span style="color: #CCC">collect()[</span><span style="color: #F08D49">0</span><span style="color: #CCC">][</span><span style="color: #F08D49">0</span><span style="color: #CCC">]</span>
<span style="color: #CCC">recent_sales_sample</span> <span style="color: #67CDCC">=</span> <span style="color: #CCC">raw_sales_df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">select(</span><span style="color: #7EC699">"recent_sales"</span><span style="color: #CCC">)</span><span style="color: #67CDCC">.</span><span style="color: #CCC">limit(</span><span style="color: #F08D49">1</span><span style="color: #CCC">)</span><span style="color: #67CDCC">.</span><span style="color: #CCC">collect()[</span><span style="color: #F08D49">0</span><span style="color: #CCC">][</span><span style="color: #F08D49">0</span><span style="color: #CCC">]</span>
<span style="color: #67CDCC">&lt;</span><span style="color: #CCC">br</span><span style="color: #67CDCC">/&gt;</span>
<span style="color: #E2777A">print</span><span style="color: #CCC">(</span><span style="color: #7EC699">f"location:        {</span><span style="color: #CCC">location_sample</span><span style="color: #7EC699">}"</span><span style="color: #CCC">)</span>
<span style="color: #E2777A">print</span><span style="color: #CCC">(</span><span style="color: #7EC699">f"payment_methods: {</span><span style="color: #CCC">payment_methods_sample</span><span style="color: #7EC699">}"</span><span style="color: #CCC">)</span>
<span style="color: #E2777A">print</span><span style="color: #CCC">(</span><span style="color: #7EC699">f"recent_sales:    {</span><span style="color: #CCC">recent_sales_sample[:</span><span style="color: #F08D49">120</span><span style="color: #CCC">]</span><span style="color: #7EC699">}..."</span><span style="color: #CCC">)</span></code></pre>
</div></div></div>
</details>

### Exercise C2: Convert with from_json

Build a DataFrame named `franchise_sales_df` that keeps `franchiseID` and `name`, and converts the three JSON columns to structured types using `from_json()` with schemas inferred by `schema_of_json()` from your samples. Print the resulting schema.

In [0]:
# C2. Convert the JSON Strings (Your Turn)
from pyspark.sql.functions import col, from_json, schema_of_json, lit

franchise_sales_df = raw_sales_df.select(
    "franchiseID",
    "name",
    from_json(col("location"), schema_of_json(lit(location_sample))).alias("location"),
    FILL_IN.alias("payment_methods"),
    FILL_IN.alias("recent_sales")
)

franchise_sales_df.printSchema()

<details>
<summary style="font-size: 1em; cursor: pointer; list-style: none; user-select: none">
<div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: center; gap: 12px">
<span>&#x25B6;</span>
<strong style="font-size: 1.1em; color: #1B5162">Expand for Solution</strong>
</div></div></summary>
<div style="border-left: 4px solid #1B5162; background: transparent; padding: 0 20px 16px 20px; border-radius: 0 0 4px 4px; margin: -16px 0 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<span style="visibility: hidden">&#x25B6;</span>
<div>
<pre style="background:#2d2d2d;border:1px solid #444;color:#cccccc;border-radius:8px;padding:16px;margin:16px 0;overflow-x:auto;font-family:Consolas,Monaco,monospace;font-size:14px;line-height:1.5"><code><span style="color: #CC99CD">from</span><span style="color: #CCC"> pyspark.sql.functions </span><span style="color: #CC99CD">import</span> <span style="color: #CCC">col,</span> <span style="color: #CCC">from_json,</span> <span style="color: #CCC">schema_of_json,</span> <span style="color: #CCC">lit</span>
<span style="color: #67CDCC">&lt;</span><span style="color: #CCC">br</span><span style="color: #67CDCC">/&gt;</span>
<span style="color: #CCC">franchise_sales_df</span> <span style="color: #67CDCC">=</span> <span style="color: #CCC">raw_sales_df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">select(</span>
    <span style="color: #7EC699">"franchiseID"</span><span style="color: #CCC">,</span>
    <span style="color: #7EC699">"name"</span><span style="color: #CCC">,</span>
    <span style="color: #CCC">from_json(col(</span><span style="color: #7EC699">"location"</span><span style="color: #CCC">),</span> <span style="color: #CCC">schema_of_json(lit(location_sample)))</span><span style="color: #67CDCC">.</span><span style="color: #CCC">alias(</span><span style="color: #7EC699">"location"</span><span style="color: #CCC">),</span>
    <span style="color: #CCC">from_json(col(</span><span style="color: #7EC699">"payment_methods"</span><span style="color: #CCC">),</span> <span style="color: #CCC">schema_of_json(lit(payment_methods_sample)))</span><span style="color: #67CDCC">.</span><span style="color: #CCC">alias(</span><span style="color: #7EC699">"payment_methods"</span><span style="color: #CCC">),</span>
    <span style="color: #CCC">from_json(col(</span><span style="color: #7EC699">"recent_sales"</span><span style="color: #CCC">),</span> <span style="color: #CCC">schema_of_json(lit(recent_sales_sample)))</span><span style="color: #67CDCC">.</span><span style="color: #CCC">alias(</span><span style="color: #7EC699">"recent_sales"</span><span style="color: #CCC">)</span>
<span style="color: #CCC">)</span>
<span style="color: #67CDCC">&lt;</span><span style="color: #CCC">br</span><span style="color: #67CDCC">/&gt;</span>
<span style="color: #CCC">franchise_sales_df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">printSchema()</span></code></pre>
</div></div></div>
</details>

## D. Working with Arrays

Now that the data is structured, let's analyze the nested sales.

### Exercise D1: Sales Volume per Franchise

Add a column `sales_count` holding the number of elements in the `recent_sales` array, select `name`, the `location.city` struct field aliased as `city`, and `sales_count`, and display the franchises ordered by `sales_count` descending.

In [0]:
# D1. Count Sales per Franchise (Your Turn)
from pyspark.sql.functions import col, array_size, desc

display(
    franchise_sales_df
    .select(
        "name",
        FILL_IN.alias("city"),
        FILL_IN.alias("sales_count")
    )
    .orderBy(FILL_IN)
)

<details>
<summary style="font-size: 1em; cursor: pointer; list-style: none; user-select: none">
<div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: center; gap: 12px">
<span>&#x25B6;</span>
<strong style="font-size: 1.1em; color: #1B5162">Expand for Solution</strong>
</div></div></summary>
<div style="border-left: 4px solid #1B5162; background: transparent; padding: 0 20px 16px 20px; border-radius: 0 0 4px 4px; margin: -16px 0 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<span style="visibility: hidden">&#x25B6;</span>
<div>
<pre style="background:#2d2d2d;border:1px solid #444;color:#cccccc;border-radius:8px;padding:16px;margin:16px 0;overflow-x:auto;font-family:Consolas,Monaco,monospace;font-size:14px;line-height:1.5"><code><span style="color: #CC99CD">from</span><span style="color: #CCC"> pyspark.sql.functions </span><span style="color: #CC99CD">import</span> <span style="color: #CCC">col,</span> <span style="color: #CCC">array_size,</span> <span style="color: #CCC">desc</span>
<span style="color: #67CDCC">&lt;</span><span style="color: #CCC">br</span><span style="color: #67CDCC">/&gt;</span>
<span style="color: #CCC">display(</span>
    <span style="color: #CCC">franchise_sales_df</span>
    <span style="color: #67CDCC">.</span><span style="color: #CCC">select(</span>
        <span style="color: #7EC699">"name"</span><span style="color: #CCC">,</span>
        <span style="color: #CCC">col(</span><span style="color: #7EC699">"location.city"</span><span style="color: #CCC">)</span><span style="color: #67CDCC">.</span><span style="color: #CCC">alias(</span><span style="color: #7EC699">"city"</span><span style="color: #CCC">),</span>
        <span style="color: #CCC">array_size(col(</span><span style="color: #7EC699">"recent_sales"</span><span style="color: #CCC">))</span><span style="color: #67CDCC">.</span><span style="color: #CCC">alias(</span><span style="color: #7EC699">"sales_count"</span><span style="color: #CCC">)</span>
    <span style="color: #CCC">)</span>
    <span style="color: #67CDCC">.</span><span style="color: #CCC">orderBy(desc(</span><span style="color: #7EC699">"sales_count"</span><span style="color: #CCC">))</span>
<span style="color: #CCC">)</span></code></pre>
</div></div></div>
</details>

### Exercise D2: Explode the Sales

Create a DataFrame named `sales_exploded_df` with one row per sale: keep `franchiseID`, `name`, and `location.country` aliased as `country`, and explode `recent_sales` into a column named `sale`. Display the first 10 rows.

In [0]:
# D2. Explode the Recent Sales Array (Your Turn)
from pyspark.sql.functions import col, explode

sales_exploded_df = franchise_sales_df.select(
    "franchiseID",
    "name",
    FILL_IN.alias("country"),
    FILL_IN.alias("sale")
)

display(sales_exploded_df.limit(10))

<details>
<summary style="font-size: 1em; cursor: pointer; list-style: none; user-select: none">
<div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: center; gap: 12px">
<span>&#x25B6;</span>
<strong style="font-size: 1.1em; color: #1B5162">Expand for Solution</strong>
</div></div></summary>
<div style="border-left: 4px solid #1B5162; background: transparent; padding: 0 20px 16px 20px; border-radius: 0 0 4px 4px; margin: -16px 0 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<span style="visibility: hidden">&#x25B6;</span>
<div>
<pre style="background:#2d2d2d;border:1px solid #444;color:#cccccc;border-radius:8px;padding:16px;margin:16px 0;overflow-x:auto;font-family:Consolas,Monaco,monospace;font-size:14px;line-height:1.5"><code><span style="color: #CC99CD">from</span><span style="color: #CCC"> pyspark.sql.functions </span><span style="color: #CC99CD">import</span> <span style="color: #CCC">col,</span> <span style="color: #CCC">explode</span>
<span style="color: #67CDCC">&lt;</span><span style="color: #CCC">br</span><span style="color: #67CDCC">/&gt;</span>
<span style="color: #CCC">sales_exploded_df</span> <span style="color: #67CDCC">=</span> <span style="color: #CCC">franchise_sales_df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">select(</span>
    <span style="color: #7EC699">"franchiseID"</span><span style="color: #CCC">,</span>
    <span style="color: #7EC699">"name"</span><span style="color: #CCC">,</span>
    <span style="color: #CCC">col(</span><span style="color: #7EC699">"location.country"</span><span style="color: #CCC">)</span><span style="color: #67CDCC">.</span><span style="color: #CCC">alias(</span><span style="color: #7EC699">"country"</span><span style="color: #CCC">),</span>
    <span style="color: #CCC">explode(col(</span><span style="color: #7EC699">"recent_sales"</span><span style="color: #CCC">))</span><span style="color: #67CDCC">.</span><span style="color: #CCC">alias(</span><span style="color: #7EC699">"sale"</span><span style="color: #CCC">)</span>
<span style="color: #CCC">)</span>
<span style="color: #67CDCC">&lt;</span><span style="color: #CCC">br</span><span style="color: #67CDCC">/&gt;</span>
<span style="color: #CCC">display(sales_exploded_df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">limit(</span><span style="color: #F08D49">10</span><span style="color: #CCC">))</span></code></pre>
</div></div></div>
</details>

### Exercise D3: Top Products by Revenue

Using `sales_exploded_df`, group by the `sale.product` struct field (aliased as `product`) and sum `sale.totalPrice` as `total_revenue`. Display the products ordered by revenue, highest first.

In [0]:
# D3. Top Products by Revenue (Your Turn)
from pyspark.sql.functions import col, sum, desc

display(
    sales_exploded_df
    .groupBy(FILL_IN.alias("product"))
    .agg(FILL_IN.alias("total_revenue"))
    .orderBy(FILL_IN)
)

<details>
<summary style="font-size: 1em; cursor: pointer; list-style: none; user-select: none">
<div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: center; gap: 12px">
<span>&#x25B6;</span>
<strong style="font-size: 1.1em; color: #1B5162">Expand for Solution</strong>
</div></div></summary>
<div style="border-left: 4px solid #1B5162; background: transparent; padding: 0 20px 16px 20px; border-radius: 0 0 4px 4px; margin: -16px 0 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<span style="visibility: hidden">&#x25B6;</span>
<div>
<pre style="background:#2d2d2d;border:1px solid #444;color:#cccccc;border-radius:8px;padding:16px;margin:16px 0;overflow-x:auto;font-family:Consolas,Monaco,monospace;font-size:14px;line-height:1.5"><code><span style="color: #CC99CD">from</span><span style="color: #CCC"> pyspark.sql.functions </span><span style="color: #CC99CD">import</span> <span style="color: #CCC">col,</span> <span style="color: #E2777A">sum</span><span style="color: #CCC">,</span> <span style="color: #CCC">desc</span>
<span style="color: #67CDCC">&lt;</span><span style="color: #CCC">br</span><span style="color: #67CDCC">/&gt;</span>
<span style="color: #CCC">display(</span>
    <span style="color: #CCC">sales_exploded_df</span>
    <span style="color: #67CDCC">.</span><span style="color: #CCC">groupBy(col(</span><span style="color: #7EC699">"sale.product"</span><span style="color: #CCC">)</span><span style="color: #67CDCC">.</span><span style="color: #CCC">alias(</span><span style="color: #7EC699">"product"</span><span style="color: #CCC">))</span>
    <span style="color: #67CDCC">.</span><span style="color: #CCC">agg(</span><span style="color: #E2777A">sum</span><span style="color: #CCC">(col(</span><span style="color: #7EC699">"sale.totalPrice"</span><span style="color: #CCC">))</span><span style="color: #67CDCC">.</span><span style="color: #CCC">alias(</span><span style="color: #7EC699">"total_revenue"</span><span style="color: #CCC">))</span>
    <span style="color: #67CDCC">.</span><span style="color: #CCC">orderBy(desc(</span><span style="color: #7EC699">"total_revenue"</span><span style="color: #CCC">))</span>
<span style="color: #CCC">)</span></code></pre>
</div></div></div>
</details>

## E. Bonus Challenge: Product Catalog per Country

Combine what you have learned: rebuild arrays from the exploded rows.

### Exercise E1: Collect the Unique Products Sold in Each Country

Using `sales_exploded_df`, group by `country` and use `collect_set()` to gather the unique products (from `sale.product`) sold in each country into a column named `products_sold`. Add a `product_count` column with the array size and display the result.

In [0]:
# E1. Unique Products per Country (Your Turn)
from pyspark.sql.functions import col, collect_set, array_size, desc

display(
    sales_exploded_df
    .groupBy("country")
    .agg(FILL_IN.alias("products_sold"))
    .withColumn("product_count", FILL_IN)
    .orderBy(desc("product_count"))
)

<details>
<summary style="font-size: 1em; cursor: pointer; list-style: none; user-select: none">
<div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: center; gap: 12px">
<span>&#x25B6;</span>
<strong style="font-size: 1.1em; color: #1B5162">Expand for Solution</strong>
</div></div></summary>
<div style="border-left: 4px solid #1B5162; background: transparent; padding: 0 20px 16px 20px; border-radius: 0 0 4px 4px; margin: -16px 0 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<span style="visibility: hidden">&#x25B6;</span>
<div>
<pre style="background:#2d2d2d;border:1px solid #444;color:#cccccc;border-radius:8px;padding:16px;margin:16px 0;overflow-x:auto;font-family:Consolas,Monaco,monospace;font-size:14px;line-height:1.5"><code><span style="color: #CC99CD">from</span><span style="color: #CCC"> pyspark.sql.functions </span><span style="color: #CC99CD">import</span> <span style="color: #CCC">col,</span> <span style="color: #CCC">collect_set,</span> <span style="color: #CCC">array_size,</span> <span style="color: #CCC">desc</span>
<span style="color: #67CDCC">&lt;</span><span style="color: #CCC">br</span><span style="color: #67CDCC">/&gt;</span>
<span style="color: #CCC">display(</span>
    <span style="color: #CCC">sales_exploded_df</span>
    <span style="color: #67CDCC">.</span><span style="color: #CCC">groupBy(</span><span style="color: #7EC699">"country"</span><span style="color: #CCC">)</span>
    <span style="color: #67CDCC">.</span><span style="color: #CCC">agg(collect_set(col(</span><span style="color: #7EC699">"sale.product"</span><span style="color: #CCC">))</span><span style="color: #67CDCC">.</span><span style="color: #CCC">alias(</span><span style="color: #7EC699">"products_sold"</span><span style="color: #CCC">))</span>
    <span style="color: #67CDCC">.</span><span style="color: #CCC">withColumn(</span><span style="color: #7EC699">"product_count"</span><span style="color: #CCC">,</span> <span style="color: #CCC">array_size(col(</span><span style="color: #7EC699">"products_sold"</span><span style="color: #CCC">)))</span>
    <span style="color: #67CDCC">.</span><span style="color: #CCC">orderBy(desc(</span><span style="color: #7EC699">"product_count"</span><span style="color: #CCC">))</span>
<span style="color: #CCC">)</span></code></pre>
</div></div></div>
</details>

## F. Knowledge Check

Check your understanding of the complex data concepts you just applied. Answer the multiple choice questions, then try the rapid fire round below them.

<div id="mcqRoot"></div>

<script>
(function() {
    var questions = [
        {
                "question": "In Exercise C2, why did we pass a sample value to schema_of_json() rather than the column itself?",
                "options": [
                        {
                                "id": "a",
                                "text": "schema_of_json() requires a literal (foldable) JSON string, not a per-row column"
                        },
                        {
                                "id": "b",
                                "text": "Columns cannot contain JSON"
                        },
                        {
                                "id": "c",
                                "text": "It is faster to parse one row than none"
                        },
                        {
                                "id": "d",
                                "text": "from_json() only accepts strings"
                        }
                ],
                "correct": "a",
                "explanation": "schema_of_json() infers one schema for the whole column, so it needs a single representative literal - here provided via lit(sample). In production you would more often declare the schema explicitly."
        },
        {
                "question": "After from_json() in Exercise C2, what is the data type of recent_sales?",
                "options": [
                        {
                                "id": "a",
                                "text": "string"
                        },
                        {
                                "id": "b",
                                "text": "array of struct elements"
                        },
                        {
                                "id": "c",
                                "text": "map of string to string"
                        },
                        {
                                "id": "d",
                                "text": "a separate table"
                        }
                ],
                "correct": "b",
                "explanation": "The JSON array of objects becomes array<struct<...>>: an array column whose elements are structs with the typed fields product, quantity, totalPrice, and dateTime."
        },
        {
                "question": "In Exercise D2, what effect did explode() have on the row count?",
                "options": [
                        {
                                "id": "a",
                                "text": "It left the row count unchanged"
                        },
                        {
                                "id": "b",
                                "text": "It reduced the rows to one per franchise"
                        },
                        {
                                "id": "c",
                                "text": "It produced one row per element of each franchise's recent_sales array"
                        },
                        {
                                "id": "d",
                                "text": "It removed rows with empty arrays only"
                        }
                ],
                "correct": "c",
                "explanation": "explode() unnests the array: each franchise row is repeated once per sale in its array, with the other columns preserved. This row multiplication is also why explode() needs care on large arrays."
        },
        {
                "question": "Why was collect_set() the right choice in Exercise E1 rather than collect_list()?",
                "options": [
                        {
                                "id": "a",
                                "text": "collect_list() cannot be used after groupBy()"
                        },
                        {
                                "id": "b",
                                "text": "We wanted each product once per country, and dropping duplicates also reduces memory and shuffle"
                        },
                        {
                                "id": "c",
                                "text": "collect_set() preserves the original sale order"
                        },
                        {
                                "id": "d",
                                "text": "collect_set() is required for string columns"
                        }
                ],
                "correct": "b",
                "explanation": "A product catalog needs each product once. collect_set() removes duplicates during aggregation, which also means less data is held in memory and shuffled compared to collect_list()."
        }
];
    
    var currentIndex = 0;
    var score = 0;
    var selectedAnswer = null;
    var showingFeedback = false;
    
    function init() {
        currentIndex = 0;
        score = 0;
        selectedAnswer = null;
        showingFeedback = false;
        render();
    }
    
    function render() {
        var root = document.getElementById('mcqRoot');
        var html = '';
        var isComplete = currentIndex >= questions.length;
        
        html += '<div style="font-family: -apple-system, BlinkMacSystemFont, Segoe UI, Roboto, sans-serif; width: 100%; margin: 10px 0; padding: 24px; background: #f5f7fa; border-radius: 12px; border: 1px solid #e0e0e0; box-sizing: border-box;">';
        
        // Header
        html += '<div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 20px;">';
        html += '<div>';
        html += '<div style="font-size: 1.4em; font-weight: 600; color: #333;">📝 Multiple Choice Quiz</div>';
        html += '<div style="font-size: 0.95em; color: #666; margin-top: 4px;">Knowledge Check</div>';
        html += '</div>';
        html += '<div style="display: flex; gap: 12px; align-items: center;">';
        html += '<span style="padding: 8px 16px; border-radius: 20px; font-weight: 600; font-size: 0.9em; background: #e8f5e9; color: #2e7d32;">Score: ' + score + '/' + questions.length + '</span>';
        html += '<span style="padding: 8px 16px; border-radius: 20px; font-weight: 600; font-size: 0.9em; background: #e3f2fd; color: #1976d2;">Q' + (currentIndex + 1) + '/' + questions.length + '</span>';
        html += '</div></div>';
        
        if (isComplete) {
            // Results
            var percentage = Math.round(score / questions.length * 100);
            html += '<div style="text-align: center; padding: 40px 20px; background: #fff; border-radius: 12px; border: 2px solid #e0e0e0;">';
            html += '<div style="font-size: 3em; margin-bottom: 16px;">' + (percentage >= 75 ? '🏆' : percentage >= 50 ? '👍' : '📚') + '</div>';
            html += '<div style="font-size: 1.8em; font-weight: 700; color: #333; margin-bottom: 8px;">Quiz Complete!</div>';
            html += '<div style="font-size: 1.2em; color: #666; margin-bottom: 24px;">You scored ' + score + ' out of ' + questions.length + ' (' + percentage + '%)</div>';
            html += '<button id="retakeBtn" style="padding: 14px 32px; background: #1976d2; color: white; border: none; border-radius: 8px; cursor: pointer; font-size: 1.1em; font-weight: 600;">Retake Quiz</button>';
            html += '</div>';
        } else {
            var q = questions[currentIndex];
            
            // Question
            html += '<div style="background: #fff; border-radius: 12px; border: 2px solid #e0e0e0; padding: 24px; margin-bottom: 20px;">';
            html += '<div style="font-size: 1.15em; color: #333; line-height: 1.6;">' + q.question + '</div>';
            html += '</div>';
            
            // Options
            html += '<div style="display: flex; flex-direction: column; gap: 12px; margin-bottom: 20px;">';
            
            q.options.forEach(function(opt) {
                var isSelected = selectedAnswer === opt.id;
                var isCorrect = opt.id === q.correct;
                var bgColor = '#fff';
                var borderColor = '#e0e0e0';
                var textColor = '#333';
                
                if (showingFeedback) {
                    if (isCorrect) {
                        bgColor = '#e8f5e9';
                        borderColor = '#4caf50';
                        textColor = '#2e7d32';
                    } else if (isSelected && !isCorrect) {
                        bgColor = '#ffebee';
                        borderColor = '#f44336';
                        textColor = '#c62828';
                    }
                } else if (isSelected) {
                    bgColor = '#e3f2fd';
                    borderColor = '#1976d2';
                    textColor = '#1565c0';
                }
                
                var cursor = showingFeedback ? 'default' : 'pointer';
                
                html += '<div class="mcq-option" data-id="' + opt.id + '" style="display: flex; align-items: center; gap: 16px; padding: 16px 20px; background: ' + bgColor + '; border: 2px solid ' + borderColor + '; border-radius: 8px; cursor: ' + cursor + '; transition: all 0.15s ease;">';
                html += '<div style="width: 32px; height: 32px; border: 2px solid ' + borderColor + '; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-weight: 700; color: ' + textColor + '; background: ' + (isSelected || (showingFeedback && isCorrect) ? borderColor : 'transparent') + '; color: ' + (isSelected || (showingFeedback && isCorrect) ? '#fff' : textColor) + ';">' + opt.id.toUpperCase() + '</div>';
                html += '<div style="flex: 1; font-size: 1em; color: ' + textColor + ';">' + opt.text + '</div>';
                if (showingFeedback && isCorrect) {
                    html += '<span style="color: #4caf50; font-size: 1.2em;">✓</span>';
                } else if (showingFeedback && isSelected && !isCorrect) {
                    html += '<span style="color: #f44336; font-size: 1.2em;">✗</span>';
                }
                html += '</div>';
            });
            
            html += '</div>';
            
            // Feedback
            if (showingFeedback) {
                var isCorrect = selectedAnswer === q.correct;
                html += '<div style="background: ' + (isCorrect ? '#e8f5e9' : '#fff3e0') + '; border: 2px solid ' + (isCorrect ? '#4caf50' : '#ff9800') + '; border-radius: 8px; padding: 16px 20px; margin-bottom: 20px;">';
                html += '<div style="font-weight: 600; color: ' + (isCorrect ? '#2e7d32' : '#e65100') + '; margin-bottom: 8px;">' + (isCorrect ? '✓ Correct!' : '✗ Not quite') + '</div>';
                html += '<div style="color: #333; font-size: 0.95em; line-height: 1.5;">' + q.explanation + '</div>';
                html += '</div>';
            }
            
            // Buttons
            html += '<div style="display: flex; gap: 12px; justify-content: flex-end;">';
            if (!showingFeedback) {
                html += '<button id="submitBtn" style="padding: 12px 28px; background: ' + (selectedAnswer ? '#4caf50' : '#ccc') + '; color: white; border: none; border-radius: 6px; cursor: ' + (selectedAnswer ? 'pointer' : 'not-allowed') + '; font-size: 1em; font-weight: 600;"' + (selectedAnswer ? '' : ' disabled') + '>Submit Answer</button>';
            } else {
                html += '<button id="nextQBtn" style="padding: 12px 28px; background: #1976d2; color: white; border: none; border-radius: 6px; cursor: pointer; font-size: 1em; font-weight: 600;">' + (currentIndex < questions.length - 1 ? 'Next Question' : 'See Results') + '</button>';
            }
            html += '</div>';
        }
        
        html += '</div>';
        
        root.innerHTML = html;
        attachEvents();
    }
    
    function attachEvents() {
        var options = document.querySelectorAll('.mcq-option');
        var submitBtn = document.getElementById('submitBtn');
        var nextQBtn = document.getElementById('nextQBtn');
        var retakeBtn = document.getElementById('retakeBtn');
        
        options.forEach(function(opt) {
            opt.addEventListener('click', function() {
                if (!showingFeedback) {
                    selectedAnswer = this.dataset.id;
                    render();
                }
            });
            
            if (!showingFeedback) {
                opt.addEventListener('mouseenter', function() {
                    if (selectedAnswer !== this.dataset.id) {
                        this.style.borderColor = '#1976d2';
                        this.style.background = '#f5f9ff';
                    }
                });
                opt.addEventListener('mouseleave', function() {
                    if (selectedAnswer !== this.dataset.id) {
                        this.style.borderColor = '#e0e0e0';
                        this.style.background = '#fff';
                    }
                });
            }
        });
        
        if (submitBtn) {
            submitBtn.addEventListener('click', function() {
                if (selectedAnswer) {
                    var q = questions[currentIndex];
                    if (selectedAnswer === q.correct) {
                        score++;
                    }
                    showingFeedback = true;
                    render();
                }
            });
        }
        
        if (nextQBtn) {
            nextQBtn.addEventListener('click', function() {
                currentIndex++;
                selectedAnswer = null;
                showingFeedback = false;
                render();
            });
        }
        
        if (retakeBtn) {
            retakeBtn.addEventListener('click', function() {
                init();
            });
        }
    }
    
    if (typeof NodeList.prototype.forEach !== 'function') {
        NodeList.prototype.forEach = Array.prototype.forEach;
    }
    
    init();
})();
</script>

<div id="trueFalseRoot"></div>

<script>
(function() {
    var questions = [
        {
                "statement": "A JSON string column and a struct column are equally efficient to query.",
                "answer": false,
                "explanation": "False! JSON strings are re-parsed by every query and are opaque to the optimizer. Structs are typed, compact, and support field pruning - convert early."
        },
        {
                "statement": "Struct fields can be accessed with dot notation, like col(\"location.city\").",
                "answer": true,
                "explanation": "Correct! Dot notation and getField() are equivalent ways to access struct fields."
        },
        {
                "statement": "explode() preserves the other columns of the row for every array element.",
                "answer": true,
                "explanation": "Correct! Each output row carries the original row's other columns alongside one array element."
        },
        {
                "statement": "collect_set() keeps duplicate values in the resulting array.",
                "answer": false,
                "explanation": "False! collect_set() returns unique values only - collect_list() is the variant that keeps duplicates."
        },
        {
                "statement": "array_size() requires exploding the array first.",
                "answer": false,
                "explanation": "False! array_size() (and array_contains(), element_at(), etc.) operate directly on the array column - no unnesting needed."
        },
        {
                "statement": "In Spark 4, the VARIANT type can store semi-structured data without a predeclared schema.",
                "answer": true,
                "explanation": "Correct! VARIANT stores semi-structured data in an efficient binary format that stays queryable while tolerating schema changes - useful when the structure is unknown or evolving."
        }
];
    
    var currentIndex = 0;
    var score = 0;
    var answered = [];
    var showingFeedback = false;
    var lastAnswer = null;
    
    function init() {
        currentIndex = 0;
        score = 0;
        answered = [];
        showingFeedback = false;
        render();
    }
    
    function render() {
        var root = document.getElementById('trueFalseRoot');
        var html = '';
        var isComplete = currentIndex >= questions.length;
        
        html += '<div style="font-family: -apple-system, BlinkMacSystemFont, Segoe UI, Roboto, sans-serif; width: 100%; margin: 10px 0; padding: 24px; background: #f5f7fa; border-radius: 12px; border: 1px solid #e0e0e0; box-sizing: border-box;">';
        
        // Header
        html += '<div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 20px;">';
        html += '<div>';
        html += '<div style="font-size: 1.4em; font-weight: 600; color: #333;">⚡ True or False: Rapid Fire</div>';
        html += '<div style="font-size: 0.95em; color: #666; margin-top: 4px;">Test your knowledge!</div>';
        html += '</div>';
        html += '<div style="display: flex; gap: 12px; align-items: center;">';
        html += '<span style="padding: 8px 16px; border-radius: 20px; font-weight: 600; font-size: 0.9em; background: #e8f5e9; color: #2e7d32;">✓ ' + score + '</span>';
        html += '<span style="padding: 8px 16px; border-radius: 20px; font-weight: 600; font-size: 0.9em; background: #e3f2fd; color: #1976d2;">' + (currentIndex) + '/' + questions.length + '</span>';
        html += '</div></div>';
        
        // Progress bar
        html += '<div style="height: 8px; background: #e0e0e0; border-radius: 4px; margin-bottom: 24px; overflow: hidden;">';
        html += '<div style="height: 100%; width: ' + (currentIndex / questions.length * 100) + '%; background: linear-gradient(90deg, #4caf50, #8bc34a); transition: width 0.3s ease;"></div>';
        html += '</div>';
        
        if (isComplete) {
            // Results screen
            var percentage = Math.round(score / questions.length * 100);
            var grade = percentage >= 80 ? '🏆 Excellent!' : percentage >= 60 ? '👍 Good job!' : '📚 Keep studying!';
            
            html += '<div style="text-align: center; padding: 40px 20px;">';
            html += '<div style="font-size: 3em; margin-bottom: 16px;">' + (percentage >= 80 ? '🎉' : percentage >= 60 ? '👏' : '💪') + '</div>';
            html += '<div style="font-size: 1.8em; font-weight: 700; color: #333; margin-bottom: 8px;">' + grade + '</div>';
            html += '<div style="font-size: 1.2em; color: #666;">You scored ' + score + ' out of ' + questions.length + ' (' + percentage + '%)</div>';
            html += '<button id="restartBtn" style="margin-top: 24px; padding: 14px 32px; background: #1976d2; color: white; border: none; border-radius: 8px; cursor: pointer; font-size: 1.1em; font-weight: 600;">Try Again</button>';
            html += '</div>';
        } else {
            // Question card
            var q = questions[currentIndex];
            
            html += '<div style="background: #fff; border: 2px solid #e0e0e0; border-radius: 12px; padding: 32px; margin-bottom: 20px;">';
            html += '<div style="font-size: 1.2em; color: #333; line-height: 1.6; text-align: center; min-height: 60px;">"' + q.statement + '"</div>';
            html += '</div>';
            
            if (showingFeedback) {
                // Feedback
                var isCorrect = lastAnswer === q.answer;
                var feedbackBg = isCorrect ? '#e8f5e9' : '#ffebee';
                var feedbackBorder = isCorrect ? '#4caf50' : '#f44336';
                var feedbackIcon = isCorrect ? '✓ Correct!' : '✗ Incorrect';
                
                html += '<div style="background: ' + feedbackBg + '; border: 2px solid ' + feedbackBorder + '; border-radius: 8px; padding: 16px 20px; margin-bottom: 20px;">';
                html += '<div style="font-weight: 600; color: ' + feedbackBorder + '; margin-bottom: 8px;">' + feedbackIcon + '</div>';
                html += '<div style="color: #333; font-size: 0.95em;">' + q.explanation + '</div>';
                html += '</div>';
                
                html += '<div style="text-align: center;">';
                html += '<button id="nextBtn" style="padding: 14px 40px; background: #1976d2; color: white; border: none; border-radius: 8px; cursor: pointer; font-size: 1.1em; font-weight: 600;">Next Question</button>';
                html += '</div>';
            } else {
                // Answer buttons
                html += '<div style="display: flex; gap: 20px; justify-content: center;">';
                html += '<button class="answer-btn" data-answer="true" style="flex: 1; max-width: 200px; padding: 20px 32px; background: #e8f5e9; border: 3px solid #4caf50; border-radius: 12px; cursor: pointer; font-size: 1.2em; font-weight: 700; color: #2e7d32; transition: all 0.15s ease;">✓ TRUE</button>';
                html += '<button class="answer-btn" data-answer="false" style="flex: 1; max-width: 200px; padding: 20px 32px; background: #ffebee; border: 3px solid #f44336; border-radius: 12px; cursor: pointer; font-size: 1.2em; font-weight: 700; color: #c62828; transition: all 0.15s ease;">✗ FALSE</button>';
                html += '</div>';
            }
        }
        
        html += '</div>';
        
        root.innerHTML = html;
        attachEvents();
    }
    
    function attachEvents() {
        var answerBtns = document.querySelectorAll('.answer-btn');
        var nextBtn = document.getElementById('nextBtn');
        var restartBtn = document.getElementById('restartBtn');
        
        answerBtns.forEach(function(btn) {
            btn.addEventListener('click', function() {
                var answer = this.dataset.answer === 'true';
                var q = questions[currentIndex];
                lastAnswer = answer;
                
                if (answer === q.answer) {
                    score++;
                }
                
                answered.push({ question: currentIndex, correct: answer === q.answer });
                showingFeedback = true;
                render();
            });
            
            btn.addEventListener('mouseenter', function() {
                this.style.transform = 'scale(1.05)';
                this.style.boxShadow = '0 4px 16px rgba(0,0,0,0.15)';
            });
            btn.addEventListener('mouseleave', function() {
                this.style.transform = 'scale(1)';
                this.style.boxShadow = 'none';
            });
        });
        
        if (nextBtn) {
            nextBtn.addEventListener('click', function() {
                currentIndex++;
                showingFeedback = false;
                render();
            });
        }
        
        if (restartBtn) {
            restartBtn.addEventListener('click', function() {
                init();
            });
        }
    }
    
    if (typeof NodeList.prototype.forEach !== 'function') {
        NodeList.prototype.forEach = Array.prototype.forEach;
    }
    
    init();
})();
</script>

## Summary

In this lab you:

- Inspected raw data whose complex values arrived as JSON strings
- Inferred schemas from sample values with `schema_of_json()` and converted the strings with `from_json()`
- Measured arrays with `array_size()` and accessed struct fields with dot notation
- Exploded the nested sales array and analyzed products and revenue
- Rebuilt per-country product catalogs with `collect_set()`

&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank" style="font-size: 1em; color: #1B5162; text-decoration: underline">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank" style="text-decoration: underline">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank" style="text-decoration: underline">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank" style="text-decoration: underline">Support</a>

<script> document.getElementById("dbx-year").textContent = new Date().getFullYear(); </script>